# EXISTS, ANY, SOME, ALL — notatki referencyjne (SQL Server)

Przykłady na modelu: `dim_Klienci` (ID_Klienta, Nazwa), `fact_Sprzedaz` (ID_Klienta, DataSprzedazy, Kwota, ID_Produktu), `dim_Produkty` (ID_Produktu, Kategoria, Cena).

## 1. `EXISTS` — podstawy i semantyka

### Składnia

```sql
WHERE EXISTS ( <podzapytanie> )
WHERE NOT EXISTS ( <podzapytanie> )
```

`EXISTS` zwraca `TRUE`, jeśli podzapytanie zwróci **przynajmniej jeden wiersz** — **nie interesuje go, co konkretnie jest w tych wierszach**, tylko sam fakt istnienia. Dlatego klasyczny styl to `SELECT 1` wewnątrz podzapytania (nie `SELECT *`) — nic z tych kolumn nie jest faktycznie odczytywane, więc pisanie `SELECT 1` jest jasnym sygnałem intencji "sprawdzam tylko istnienie", a silnik i tak zoptymalizuje to identycznie niezależnie od tego, co wpiszesz po `SELECT`.

### Przykład — klienci, którzy mieli chociaż jedną transakcję powyżej 10 000

```sql
SELECT k.ID_Klienta, k.Nazwa
FROM dim_Klienci k
WHERE EXISTS (
    SELECT 1
    FROM fact_Sprzedaz s
    WHERE s.ID_Klienta = k.ID_Klienta
      AND s.Kwota > 10000
);
```

Podzapytanie wewnątrz `EXISTS` — dokładnie jak przy `APPLY` — odwołuje się do `k` (bieżącego wiersza zewnętrznego zapytania) przez `s.ID_Klienta = k.ID_Klienta`. To jest tzw. **skorelowane podzapytanie** (correlated subquery).

## 2. `NOT EXISTS` — anti-join, i dlaczego jest bezpieczniejszy niż `NOT IN`

### Przykład — klienci bez żadnej transakcji

```sql
SELECT k.ID_Klienta, k.Nazwa
FROM dim_Klienci k
WHERE NOT EXISTS (
    SELECT 1 FROM fact_Sprzedaz s WHERE s.ID_Klienta = k.ID_Klienta
);
```

### Krytyczna pułapka `NOT IN` z `NULL` — ta różnica ma realne znaczenie

To jest jeden z najbardziej podstępnych, cichych błędów w SQL — wart zapamiętania na zawsze:

```sql
-- WYGLĄDA na równoważne powyższemu NOT EXISTS, ALE NIE JEST BEZPIECZNE
SELECT k.ID_Klienta, k.Nazwa
FROM dim_Klienci k
WHERE k.ID_Klienta NOT IN (
    SELECT s.ID_Klienta FROM fact_Sprzedaz s
);
```

**Jeśli `fact_Sprzedaz.ID_Klienta` zawiera choćby jeden `NULL`** (np. transakcja bez przypisanego klienta — błąd danych albo świadomie dopuszczalny brak), **całe zapytanie `NOT IN` zwróci ZERO wierszy** — nie błąd, nie ostrzeżenie, po prostu cichą, pustą listę wyników, mimo że realnie są klienci bez transakcji.

**Dlaczego tak się dzieje:** `NOT IN (SELECT ...)` semantycznie rozwija się do serii porównań `<> wartość1 AND <> wartość2 AND ...`. Porównanie `cokolwiek <> NULL` w SQL **nie zwraca `TRUE` ani `FALSE`, tylko `UNKNOWN`** (trójwartościowa logika SQL) — a `UNKNOWN` w łańcuchu `AND` zamienia **cały wynik** na `UNKNOWN`, co dla `WHERE` jest równoznaczne z odrzuceniem wiersza. Jeden `NULL` w podzapytaniu psuje cały wynik dla wszystkich wierszy, nie tylko dla tego jednego przypadku.

**`NOT EXISTS` nie ma tego problemu** — sprawdza istnienie dopasowania przez `=`, a nie buduje listy do porównania przez `<>`, więc `NULL`-e w `fact_Sprzedaz.ID_Klienta` nie wpływają na wynik dla innych klientów.

**Reguła praktyczna, bez wyjątków: używaj `NOT EXISTS` zamiast `NOT IN` z podzapytaniem, zawsze, chyba że masz absolutną pewność (np. przez `NOT NULL` constraint w bazie), że kolumna w podzapytaniu nigdy nie zawiera `NULL`.** To nie jest kwestia stylu — to kwestia poprawności, którą łatwo przeoczyć, bo `NOT IN` "działa" w testach na czystych danych testowych i psuje się dopiero na produkcji, gdy pojawi się pierwszy `NULL`.

## 3. `EXISTS` vs `IN` vs `JOIN`+`DISTINCT` — wydajność i kiedy co

Trzy sposoby na to samo pytanie: "klienci, którzy kupili produkty z kategorii Elektronika":

```sql
-- Wersja EXISTS
SELECT k.ID_Klienta, k.Nazwa
FROM dim_Klienci k
WHERE EXISTS (
    SELECT 1 FROM fact_Sprzedaz s
    JOIN dim_Produkty p ON p.ID_Produktu = s.ID_Produktu
    WHERE s.ID_Klienta = k.ID_Klienta AND p.Kategoria = 'Elektronika'
);

-- Wersja IN
SELECT k.ID_Klienta, k.Nazwa
FROM dim_Klienci k
WHERE k.ID_Klienta IN (
    SELECT s.ID_Klienta FROM fact_Sprzedaz s
    JOIN dim_Produkty p ON p.ID_Produktu = s.ID_Produktu
    WHERE p.Kategoria = 'Elektronika'
);

-- Wersja JOIN + DISTINCT
SELECT DISTINCT k.ID_Klienta, k.Nazwa
FROM dim_Klienci k
JOIN fact_Sprzedaz s ON s.ID_Klienta = k.ID_Klienta
JOIN dim_Produkty p ON p.ID_Produktu = s.ID_Produktu
WHERE p.Kategoria = 'Elektronika';
```

**Poprawność:** wszystkie trzy dają ten sam wynik **tutaj**, bo `IN` z listą wartości (nie `NOT IN`) nie ma problemu z `NULL` analogicznego do sekcji 2 — `NULL` w liście `IN` po prostu nigdy nie pasuje, nie psuje reszty porównań (`OR` z `UNKNOWN` nie "zaraża" całego wyniku tak, jak `AND` przy `NOT IN`).

**Wydajność:** w nowoczesnym SQL Server optymalizator zapytań **zwykle** przekształca wszystkie trzy formy do bardzo podobnego, jeśli nie identycznego, planu wykonania (semi-join) — różnice bywają nieistotne. To nie jest reguła bez wyjątków (zależy od wersji silnika, statystyk, indeksów), więc przy realnym podejrzeniu problemu wydajnościowego **sprawdź plan wykonania** (`SET STATISTICS IO ON` + Actual Execution Plan), zamiast zakładać z góry, która forma jest szybsza — dokładnie ta sama zasada, którą stosowaliśmy przy porównaniach w DAX.

**Kiedy wybrać którą formę — kwestia czytelności, nie tylko wydajności:**
- `EXISTS` — najlepszy, gdy interesuje Cię wyłącznie **fakt istnienia** dopasowania, nie potrzebujesz żadnych kolumn z dopasowanej tabeli w wyniku głównym. Naturalnie odporny na pułapkę `NULL`.
- `IN` — czytelny przy prostej liście wartości z jednej kolumny, zwłaszcza jeśli podzapytanie jest proste (bez dodatkowych `JOIN`ów).
- `JOIN`+`DISTINCT` — jedyny wybór, gdy **faktycznie potrzebujesz** kolumn z dopasowanej tabeli w wyniku (nie tylko sprawdzasz istnienie) — ale pamiętaj o `DISTINCT`, inaczej dostaniesz duplikaty klientów (po jednym na każdą pasującą transakcję), co jest częstym błędem przy zamianie `EXISTS` na `JOIN` bez zastanowienia.

## 4. `ANY` / `SOME` — porównanie skalara z wynikami podzapytania

`ANY` i `SOME` to **dokładne synonimy** w SQL Server (identyczna semantyka, różne słowo kluczowe — historyczne dziedzictwo standardu SQL) — używaj tego, które Twój zespół preferuje stylistycznie, bez różnicy funkcjonalnej.

### Składnia

```sql
WHERE <wyrażenie> <operator_porównania> ANY ( <podzapytanie> )
WHERE <wyrażenie> <operator_porównania> SOME ( <podzapytanie> )
```

`<wyrażenie> > ANY (...)` zwraca `TRUE`, jeśli `<wyrażenie>` jest większe od **przynajmniej jednej** wartości zwróconej przez podzapytanie.

### Przykład — produkty droższe niż przynajmniej jeden produkt w kategorii "Budżetowa"

```sql
SELECT p.ID_Produktu, p.Nazwa, p.Cena
FROM dim_Produkty p
WHERE p.Cena > ANY (
    SELECT p2.Cena FROM dim_Produkty p2 WHERE p2.Kategoria = 'Budżetowa'
);
```

To zwróci każdy produkt, który jest droższy niż **najtańszy** produkt w kategorii "Budżetowa" — bo wystarczy pokonać choć jedną wartość z listy, żeby warunek `> ANY` był spełniony. W praktyce `> ANY (...)` jest równoważne `> (SELECT MIN(...))`, a `< ANY (...)` równoważne `< (SELECT MAX(...))` — często czytelniej jest po prostu napisać to przez `MIN`/`MAX`, chyba że lista wartości i tak jest Ci potrzebna do czegoś innego w tym samym zapytaniu.

**Pułapka interpretacyjna — `= ANY` to w praktyce to samo co `IN`:**

```sql
WHERE p.Kategoria = ANY ( SELECT DISTINCT Kategoria FROM kategorie_promocyjne )
-- równoważne:
WHERE p.Kategoria IN ( SELECT DISTINCT Kategoria FROM kategorie_promocyjne )
```

`= ANY (...)` i `IN (...)` to dokładnie ta sama semantyka — `IN` jest po prostu starszym, bardziej rozpowszechnionym zapisem tego samego mechanizmu. Nie ma powodu używać `= ANY` zamiast `IN` poza czystą preferencją stylistyczną.

## 5. `ALL` — warunek musi być spełniony względem WSZYSTKICH wartości

### Składnia

```sql
WHERE <wyrażenie> <operator_porównania> ALL ( <podzapytanie> )
```

`<wyrażenie> > ALL (...)` zwraca `TRUE` tylko, jeśli `<wyrażenie>` jest większe od **każdej** wartości zwróconej przez podzapytanie — czyli w praktyce większe od **maksimum**.

### Przykład — produkt droższy niż wszystkie produkty w kategorii "Budżetowa"

```sql
SELECT p.ID_Produktu, p.Nazwa, p.Cena
FROM dim_Produkty p
WHERE p.Cena > ALL (
    SELECT p2.Cena FROM dim_Produkty p2 WHERE p2.Kategoria = 'Budżetowa'
);
```

Równoważne `p.Cena > (SELECT MAX(p2.Cena) FROM dim_Produkty p2 WHERE p2.Kategoria = 'Budżetowa')` — tu również, jeśli i tak potrzebujesz tylko tego jednego porównania, `MAX` jest zwykle czytelniejszy. `ALL`/`ANY` zyskują na wartości głównie w zapytaniach generowanych dynamicznie/programistycznie, albo gdy podzapytanie ma dodatkową złożoność, której nie da się łatwo sprowadzić do jednego `MIN`/`MAX`.

### Pułapka — pusty zbiór po prawej stronie `ALL` daje zawsze `TRUE`

Jeśli podzapytanie w `ALL (...)` **nie zwróci żadnego wiersza** (np. kategoria "Budżetowa" w ogóle nie istnieje w danych), warunek `> ALL (pusty zbiór)` jest **logicznie zawsze prawdziwy** ("większe od każdej wartości pustego zbioru" — warunek pusty jest trywialnie spełniony) — **wszystkie** wiersze przejdą przez filtr, nie żaden. To częsty, zaskakujący błąd, gdy podzapytanie może w pewnych warunkach (np. filtr na nieistniejącą wartość) zwrócić pusty zbiór. Warto to świadomie przetestować, jeśli kategoria/warunek filtra pochodzi z parametru, który teoretycznie mógłby nie dać żadnego dopasowania.

## 6. Zestawienie — `EXISTS`/`IN`/`ANY`/`ALL` obok siebie

| Chcesz sprawdzić | Konstrukcja | Uwaga |
|---|---|---|
| Czy istnieje przynajmniej jedno dopasowanie (bez potrzeby wartości) | `EXISTS (...)` | Najbezpieczniejszy, odporny na `NULL` |
| Czy NIE istnieje żadne dopasowanie | `NOT EXISTS (...)` | Zawsze zamiast `NOT IN` przy podzapytaniu — bezpieczeństwo `NULL` |
| Wartość jest jedną z listy zwróconej przez podzapytanie | `IN (...)` | Bezpieczne (w przeciwieństwie do `NOT IN`) |
| Wartość jest większa/mniejsza od co najmniej jednej wartości z listy | `> ANY (...)` / `> SOME (...)` | Równoważne `> (SELECT MIN(...))` przy operatorze `>` |
| Wartość jest większa/mniejsza od WSZYSTKICH wartości z listy | `> ALL (...)` | Równoważne `> (SELECT MAX(...))` przy operatorze `>`; uważaj na pusty zbiór (zawsze `TRUE`) |
| Potrzebujesz też kolumn z dopasowanej tabeli w wyniku | `JOIN` (+ `DISTINCT`, jeśli relacja 1:N) | `EXISTS`/`IN` nie dają dostępu do kolumn podzapytania w `SELECT` głównym |

**Złota zasada na koniec: `NOT IN` z podzapytaniem to jedna z niewielu konstrukcji SQL, której warto unikać niemal bez wyjątków — zastępuj ją `NOT EXISTS`, zanim jeden `NULL` w produkcyjnych danych po cichu wyzeruje Ci wynik zapytania.**